In [ ]:
# !pip install opencv-python matplotlib

In [ ]:
# Import das bibliotecas necessárias

import matplotlib.pyplot as plt
import cv2 as cv
import os
import random

In [ ]:
def plotar(imagens, titulos=None, _ncols=3):
    """
    Plota uma lista de imagens em uma grade dinâmica.
    imagens: lista de arrays (imagens).
    titulos: lista opcional de strings com títulos para cada imagem.
    """
    import matplotlib.pyplot as plt
    import cv2 as cv
    import math

    n_imagens = len(imagens)

    if n_imagens == 0:
        print("Nenhuma imagem fornecida.")
        return

    # Define grid automática (ex: até 4 imagens -> 2x2)
    ncols = min(_ncols, n_imagens)
    nrows = math.ceil(n_imagens / ncols)

    # Calcula tamanho da figura baseado no total de imagens
    figsize = (ncols * 15, nrows * 10)
    plt.figure(figsize=figsize)

    for idx, imagem in enumerate(imagens):
        plt.subplot(nrows, ncols, idx + 1)

        if len(imagem.shape) == 2:
            plt.imshow(imagem, cmap='gray')
        else:
            plt.imshow(cv.cvtColor(imagem, cv.COLOR_BGR2RGB))

        # Define título se existir
        if titulos and idx < len(titulos):
            plt.title(titulos[idx], fontsize=35)
        else:
            plt.title(f'Imagem {idx+1}')

        plt.axis('off')

    plt.tight_layout()
    plt.show()


In [ ]:
def plotarColuna(imagens, titulos=None, _ncols=3):
    """
    Plota uma lista de imagens ou histogramas em uma grade dinâmica.
    imagens: lista de arrays (imagens 2D/3D ou histogramas 1D/2D).
    titulos: lista opcional de strings com títulos para cada item.
    """
    import matplotlib.pyplot as plt
    import cv2 as cv
    import numpy as np
    import math

    n_imagens = len(imagens)

    if n_imagens == 0:
        print("Nenhuma imagem fornecida.")
        return

    ncols = min(_ncols, n_imagens)
    nrows = math.ceil(n_imagens / ncols)

    figsize = (ncols * 15, nrows * 10)
    plt.figure(figsize=figsize)

    for idx, item in enumerate(imagens):
        plt.subplot(nrows, ncols, idx + 1)

        # Converte histogramas 2D para 1D
        if isinstance(item, np.ndarray) and (len(item.shape) == 1 or (len(item.shape) == 2 and item.shape[1] == 1)):
            # hist = item.ravel()
            # hist_norm = hist / hist.max()  # normaliza
            # plt.bar(range(len(hist_norm)), hist_norm, width=1.0, color='gray')
            # plt.xlim([0, 256])
            # plt.ylim([0, 1])
            # plt.grid(alpha=0.3)
            # plt.xlabel('Intensidade', fontsize=14)
            # plt.ylabel('Frequência normalizada', fontsize=14)
            
            hist = item.ravel()  # achata para 1D
            plt.plot(hist)
            plt.title(titulos[idx] if titulos and idx < len(titulos) else f'Histograma {idx+1}', fontsize=25)
            plt.xlim([0, 256])
            plt.xlabel('Intensidade')
            plt.ylabel('Frequência')
        else:
            if len(item.shape) == 2:
                plt.imshow(item, cmap='gray')
            else:
                plt.imshow(cv.cvtColor(item, cv.COLOR_BGR2RGB))
            # plt.title(titulos[idx] if titulos and idx < len(titulos) else f'Imagem {idx+1}', fontsize=25)
            plt.axis('off')
            
        # Define título se existir
        if titulos and idx < len(titulos):
            plt.title(titulos[idx], fontsize=35)
        else:
            plt.title(f'Imagem {idx+1}')

    plt.tight_layout()
    plt.show()


In [ ]:
imagens_carregadas = []

def carregar_path_imagem(caminho):
    # Percorre o diretório especificado e carrega as imagens
    for filename in os.listdir(caminho):
        # se for uma pasta acessa para verificar se tem alguma imagem.
        if os.path.isdir(os.path.join(caminho, filename)):
            carregar_path_imagem(os.path.join(caminho, filename))
        # Verifica se o arquivo é uma imagem
        if not filename.endswith(('.png', '.jpg', '.jpeg', '.bmp')):
            continue
        
        caminho_completo = os.path.join(caminho, filename)
        classe = os.path.basename(os.path.dirname(caminho_completo))
        imagens_carregadas.append(caminho_completo)
            
    return imagens_carregadas

In [ ]:
# Lê algumas imagens para teste cv.imread()

def carregar_imagem(numero_amostras):
    caminho_todas_imagens = carregar_path_imagem(r'..\dados\classes')
    total = len(caminho_todas_imagens)
    imagens_carregadas = []

    for i in range(numero_amostras):
        # seleciona aleatoriamente um número entre 0 e a quantidade total de imagens.    
        x = random.randint(0, len(caminho_todas_imagens))
        # Carrega a imagem usando o opencv e adiciona em forma de tupla a imagem na posição 0 e o nome da imagem na posição 1 (o nome é usado para legenda depois.)
        nome_imagem = os.path.basename(caminho_todas_imagens[x])
        classe_imagem = os.path.basename(os.path.dirname(caminho_todas_imagens[x]))
        imagens_carregadas.append((nome_imagem, cv.imread(caminho_todas_imagens[x]), classe_imagem))
        
    return imagens_carregadas

In [ ]:
#carrega as imagens e adiciona em uma lista
NUMERO_AMOSTRAS = 15

imagens_originais = carregar_imagem(NUMERO_AMOSTRAS)
# Faz a conversão para monocromático na função carregar_imagem, quem define isso é o segundo parametro da função
imagens_monocromaticas = [(nome, cv.cvtColor(imagem, cv.COLOR_BGR2GRAY), classe) for nome, imagem, classe in imagens_originais]


In [ ]:
# img = cv.imread(r'.\IBAGENS\2.jpg', cv.IMREAD_GRAYSCALE)
# hist_original = cv.calcHist([img], [0], None, [256], [0,256])

# img_eq = cv.equalizeHist(img)
# hist_eq = cv.calcHist([img_eq], [0], None, [256], [0,256])

itens = []
for nome, imagem, _ in imagens_monocromaticas:
    hist_original = cv.calcHist([imagem], [0], None, [256], [0,256])
    img_eq = cv.equalizeHist(imagem)
    hist_eq = cv.calcHist([img_eq], [0], None, [256], [0,256])
    itens.extend([imagem, hist_original, img_eq, hist_eq])

plotarColuna(itens, None, 4)

In [ ]:
itens = []
for nome, imagem, _ in imagens_monocromaticas:
    hist_original = cv.calcHist([imagem], [0], None, [256], [0,256])
    clahe = cv.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    img_clahe = clahe.apply(imagem)
    hist_clare = cv.calcHist([img_clahe], [0], None, [256], [0,256])
    # itens.extend([imagem, img_clahe, hist_original, hist_clare])
    itens.extend([imagem, img_clahe])

plotarColuna(itens, None, 2)

In [ ]:
# Plota as imagens convertidas para monocromáticas    
plotar([imagem for nome, imagem, _ in imagens_originais], [f'Original {str(i+1)}' for i in range(len(imagens_originais))], NUMERO_AMOSTRAS)
plotar([imagem for nome, imagem, _ in imagens_monocromaticas], [f'Monocromatica {str(i+1)}' for i in range(len(imagens_monocromaticas))], NUMERO_AMOSTRAS)

In [ ]:
# Normaliza as imagens monocromáticas dividindo por 2

imagens_normalizadas = []
# Percorre as imagens monocromáticas e normaliza cada uma
# A normalização é feita dividindo o valor de cada pixel por 2
# As imagens normalizadas são armazenadas em uma nova lista
# a normalização é feita para reduzir o brilho da imagem, tornando-a mais adequada para visualização ou processamento posterior
# Alem disso a normalização pode ajudar a evitar problemas de saturação em imagens muito brilhantes
# Isso é especialmente útil em processamento de imagens, onde valores muito altos podem causar distorções
for nome, imagem, classe in imagens_monocromaticas:
    imagem_normalizada = cv.divide(imagem, 2)
    imagens_normalizadas.append((nome, imagem_normalizada, classe))
    
# Plota as imagens normalizadas
plotar([imagem for nome, imagem, classe in imagens_monocromaticas], ['Monocromatica: ' + str(i+1) for i in range(len(imagens_monocromaticas))], NUMERO_AMOSTRAS)
plotar([imagem for nome, imagem, classe in imagens_normalizadas], ['Normalizada: ' + str(i+1) for i in range(len(imagens_normalizadas))], NUMERO_AMOSTRAS)
    

In [ ]:
# Ajuste de brilho

imagens_brilho = []
for nome, imagem, classe in imagens_normalizadas:    
    imagem_ajustada = cv.add(imagem, 60)  # Aumenta o brilho em 50
    imagens_brilho.append((nome, imagem_ajustada, classe))
    
    
#plota as imagens com brilho ajustado
plotar([imagem for nome, imagem, classe in imagens_normalizadas], ['Normalizada: ' + str(i+1) for i in range(len(imagens_normalizadas))], NUMERO_AMOSTRAS)
plotar([imagem for nome, imagem, classe in imagens_brilho], ['Brilho 60: ' + str(i+1) for i in range(len(imagens_brilho))], NUMERO_AMOSTRAS)

In [ ]:
# Aqui estou realizando o teste de limiarização das imagens com brilho ajustado
# O teste de limiarização é feito com diferentes valores de limiar, adicionado a uma lista e depois plotado para comparação
imagens_limiarizadas = []
for nome, imagem, classe in imagens_brilho:
    imagens_limiarizadas.append(('Original', imagem)) 
    _, imagem_limiarizada = cv.threshold(imagem, 107, 255, cv.THRESH_BINARY)
    imagens_limiarizadas.append(('107', imagem_limiarizada)) 
    _, imagem_limiarizada = cv.threshold(imagem, 117, 255, cv.THRESH_BINARY)
    imagens_limiarizadas.append(('117', imagem_limiarizada)) 
    _, imagem_limiarizada = cv.threshold(imagem, 124, 255, cv.THRESH_BINARY)
    imagens_limiarizadas.append(('124', imagem_limiarizada)) 
    _, imagem_limiarizada = cv.threshold(imagem, 127, 255, cv.THRESH_BINARY)
    imagens_limiarizadas.append(('127', imagem_limiarizada)) 
    
plotar([imagem for nome, imagem in imagens_limiarizadas], [nome for nome, _ in imagens_limiarizadas], _ncols=5)    

In [ ]:
# Define o dataset a ser utilizado com a limiarização escolhida conforme comparação de imagens feitas anteriormente
# Em testes validei que o valor de 124 me pareceu o mais adequado para as 3 imagens de amostragem.
imagens_limiarizadas = []
for nome, imagem, _ in imagens_brilho:
    _, imagem_limiarizada = cv.threshold(imagem, 124, 255, cv.THRESH_BINARY)
    imagens_limiarizadas.append((nome, imagem_limiarizada))     
    
plotar([imagem for nome, imagem in imagens_limiarizadas], ['Limiarizada 124: ' + str(i+1) for i in range(len(imagem_limiarizada))], NUMERO_AMOSTRAS)

In [ ]:
# Inversão dos tons de cinza para negativo (operador lógico NOT)
imagens_negativas = []
for nome, imagem in imagens_limiarizadas:
    imagem_negativa = cv.bitwise_not(imagem)
    imagens_negativas.append((nome, imagem_negativa))
    
# Plota as imagens negativas
plotar([imagem for nome, imagem in imagens_negativas], titulos=['Negativa: ' + str(i+1) for i in range(len(imagens_negativas))])

In [ ]:
# Teste logico das imagens em
imagens_logicos = []
ja_usados = set()

def operacoes_logicas(ds1, ds_destino):
    for idx, (nome, imagem) in enumerate(ds1):
        for i in range(len(ds1)): 
            if idx == i or (idx, i) in ja_usados or (i, idx) in ja_usados:
                continue
            
            imagem_and = cv.bitwise_and(imagem, ds1[i][1])
            imagem_or = cv.bitwise_or(imagem, ds1[i][1])
            imagem_xor = cv.bitwise_xor(imagem, ds1[i][1])
            
            ds_destino.append((f'Imagem {idx+1}', imagem))
            ds_destino.append((f'Imagem {i+1}', ds1[i][1]))
            ds_destino.append((f'AND Imagem {idx+1}_{i+1}', imagem_and))
            ds_destino.append((f'OR Imagem {idx+1}_{i+1}', imagem_or))
            ds_destino.append((f'XOR Imagem {idx+1}_{i+1}', imagem_xor))
            ja_usados.add((idx, i))
            
            # print({imagens_limiarizadas[i][0]})
            cv.imwrite(rf'..\Atividade_1\Imagens\Operadores\AND_{str(idx+1)}_{str(i+1)}.jpg', imagem_and)
            cv.imwrite(rf'..\Atividade_1\Imagens\Operadores\OR_{str(idx+1)}_{str(i+1)}.jpg', imagem_or)
            cv.imwrite(rf'..\Atividade_1\Imagens\Operadores\XOR_{str(idx+1)}_{str(i+1)}.jpg', imagem_xor)
            

nova_resolucao = (1280, 720)
imagens_negativas = [(nome, cv.resize(img, nova_resolucao)) for nome, img in imagens_negativas]
operacoes_logicas(imagens_negativas, imagens_logicos)
# subtracao_imagens(imagens_limiarizadas, imagens_negativas, imagens_subtracao)

plotar([imagem for nome, imagem in imagens_logicos], titulos=[nome for nome, imagem in imagens_logicos], _ncols=5)

In [ ]:
# Subtração de imagens
# A subtração de imagens é uma operação que calcula a diferença entre duas imagens pixel as pixel.
# Essa operação é útil para destacar diferenças entre imagens, como mudanças em uma cena ao longo do tempo ou diferenças entre duas imagens capturadas sob condições diferentes.
imagens_subtracao = []
ja_usados = set()

def subtracao_imagens(ds1, ds_destino):
    for idx, (nome, imagem) in enumerate(ds1):
        for i in range(len(ds1)): 
            if idx == i or (idx, i) in ja_usados or (i, idx) in ja_usados:
                continue
            
            imagem_subtracao = cv.subtract(imagem, ds1[i][1])
            imagem_subtracao_invertida = cv.subtract(ds1[i][1], imagem)
            
            ds_destino.append((f'Imagem {idx+1}', imagem))
            ds_destino.append((f'Imagem {i+1}', ds1[i][1]))
            ds_destino.append((f'Imagem {idx+1}_{i+1}', imagem_subtracao))
            ds_destino.append((f'Imagem {i+1}_{idx+1}', imagem_subtracao_invertida))
            ja_usados.add((idx, i))
            

subtracao_imagens(imagens_negativas, imagens_subtracao)
plotar([imagem for nome, imagem in imagens_subtracao], titulos=[nome for nome, imagem in imagens_subtracao], _ncols=4)